# Working with a real world data-set using SQL and Python



## Objectives

*   Understand the dataset for Chicago Public School level performance
*   Store the dataset in SQLite database.
*   Retrieve metadata about tables and columns and query data from mixed case columns
*   Solve example problems to practice my SQL skills including using built-in database functions


In [ ]:
#Connecting to the Database

import csv, sqlite3
con= sqlite3.connect("chicago_data.db")
cur = con.cursor()
    

In [ ]:
#load the sql extension
%load_ext sql

In [ ]:
# using the SQL magic command %sql to connect to an SQLite database named "chicago_data.db"
%sql sqlite:///chicago_data.db

### Store the dataset in a Table

##### In many cases the dataset to be analyzed is available as a .CSV (comma separated values) file, perhaps on the internet. To analyze the data 
##### using SQL, it first needs to be stored in the database.

##### We will first read the csv files  from the given url  into pandas dataframes

##### Next we will be using the  df.to_sql() function to convert each csv file  to a table in sqlite  with the csv data loaded in it.


In [ ]:
import pandas
df = pandas.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoCensusData.csv")
df.to_sql("CENSUS_DATA", con, if_exists='replace', index=False,method="multi")

df = pandas.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoCrimeData.csv")
df.to_sql("CHICAGO_CRIME_DATA", con, if_exists='replace', index=False, method="multi")

df = pandas.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoPublicSchools.csv")
df.to_sql("CHICAGO_PUBLIC_SCHOOLS_DATA", con, if_exists='replace', index=False, method="multi")

df = pandas.read_csv('https://data.cityofchicago.org/resource/jcxq-k9xf.csv')
df.to_sql("chicago_socioeconomic_data", con, if_exists='replace', index=False,method="multi")


 ### Query the database system catalog to retrieve table metadata

##### You can verify that the table creation was successful by retrieving the list of all tables in your schema and checking whether the SCHOOLS table was created


In [ ]:
%sql SELECT name FROM sqlite_master WHERE type='table'

### Query the database system catalog to retrieve column metadata

##### The SCHOOLS table contains a large number of columns. How many columns does this table have?


In [ ]:
%sql SELECT COUNT(name) FROM PRAGMA_TABLE_INFO ('CHICAGO_PUBLIC_SCHOOLS_DATA');

In [ ]:
#Now retrieve the the list of columns in SCHOOLS table and their column type (datatype) and length.

%sql SELECT name, type, length(type) FROM PRAGMA_TABLE_INFO('CHICAGO_PUBLIC_SCHOOLS_DATA');

In [ ]:
#How many Elementary Schools are in the dataset?

%sql select count(*) FROM CHICAGO_PUBLIC_SCHOOLS_DATA WHERE [Elementary, Middle, or High School]='ES';

In [ ]:
#What is the highest Safety Score?
%sql select max(SAFETY_SCORE)  as MAX_SAFETY_SCORE from CHICAGO_PUBLIC_SCHOOLS_DATA;

In [ ]:
#Which schools have highest Safety Score?
%sql select NAME_OF_SCHOOL,SAFETY_SCORE from CHICAGO_PUBLIC_SCHOOLS_DATA \
where SAFETY_SCORE=(SELECT max(SAFETY_SCORE) from CHICAGO_PUBLIC_SCHOOLS_DATA);

In [ ]:
#What are the top 10 schools with the highest "Average Student Attendance"??

%sql select Name_of_School, Average_Student_Attendance from CHICAGO_PUBLIC_SCHOOLS_DATA order by Average_Student_Attendance desc limit 10;

In [ ]:
#Retrieve the list of 5 Schools with the lowest Average Student Attendance sorted in ascending order based on attendance

%sql SELECT Name_of_School, Average_Student_Attendance  \
     from CHICAGO_PUBLIC_SCHOOLS_DATA \
     order by Average_Student_Attendance \
     LIMIT 5

In [ ]:
#removing the '%' sign from the above result set for Average Student Attendance column


%sql SELECT Name_of_School, REPLACE(Average_Student_Attendance, '%', '') as averege\
     from CHICAGO_PUBLIC_SCHOOLS_DATA \
     order by Average_Student_Attendance \
     LIMIT 5

In [ ]:
#Which Schools have Average Student Attendance lower than 70%?

%sql SELECT Name_of_School, Average_Student_Attendance  \
     from CHICAGO_PUBLIC_SCHOOLS_DATA \
     where CAST ( REPLACE(Average_Student_Attendance, '%', '') AS DOUBLE ) < 70 \
     order by Average_Student_Attendance

In [ ]:
#Get the total College Enrollment for each Community Area

%sql select Community_Area_Name, sum(College_Enrollment) AS TOTAL_ENROLLMENT \
   from CHICAGO_PUBLIC_SCHOOLS_DATA \
   group by Community_Area_Name 

In [ ]:
#Get the 5 Community Areas with the least total College Enrollment sorted in ascending order

%sql select Community_Area_Name, sum(College_Enrollment) AS TOTAL_ENROLLMENT \
   from CHICAGO_PUBLIC_SCHOOLS_DATA \
   group by Community_Area_Name \
   order by TOTAL_ENROLLMENT asc \
   LIMIT 5 

In [ ]:
#List 5 schools with lowest safety score.

%sql SELECT name_of_school, safety_score \
FROM CHICAGO_PUBLIC_SCHOOLS_DATA  where safety_score !='None' \
ORDER BY safety_score \
LIMIT 5

In [ ]:
#Get the hardship index for the community area which has College Enrollment of 4368



%sql select hardship_index from CENSUS_DATA CD, CHICAGO_PUBLIC_SCHOOLS_DATA CPS \
where CD.community_area_number = CPS.COMMUNITY_AREA_NUMBER \
and college_enrollment = 4368

In [ ]:
#Get the hardship index for the community area which has the highest value for College Enrollment

%sql select community_area_number, community_area_name, hardship_index from CENSUS_DATA \
   where community_area_number in \
   ( select community_area_number from CHICAGO_PUBLIC_SCHOOLS_DATA order by college_enrollment desc limit 1 )